In [ ]:
from datetime import datetime
import numpy as np
import pandas as pd
import preprocessing_dataset
import cross_validation
import os

In [ ]:
import importlib
grid = '4x3'
eval_mode = 'cv'
path = os.path.dirname(os.getcwd())
cme_list, hss_list, enhancement_list = preprocessing_dataset.read_cme_hss_list(path)
X, Y = preprocessing_dataset.read_ml_dataset(path, grid)

In [ ]:
if eval_mode == 'cv':
    cv_range = Y.index < datetime(2020,1,1)
    Y = Y.iloc[cv_range]
    X = X.iloc[cv_range,:]
    data_split = cross_validation.cross_validation_split(Y)

data_split_without_cmes = cross_validation.delete_cmes_from_data_split(data_split, cme_list)
data_split = [[data_split_without_cmes[0], data_split_without_cmes[1]],
              [data_split[0], data_split[1]]]
data_split = pd.DataFrame(data_split, dtype=object, index=['no_cme', 'with_cme'], columns=['train', 'test'])

In [ ]:
import xgboost as xgb
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np

xgb_params = {
    'n_estimators': 500,
    'max_depth': 3,
    'learning_rate': 0.051212530070895795,
    'subsample': 0.505483952118673,
    'colsample_bytree': 0.6326585694513049,
    'min_child_weight': 4,
    'gamma': 2.243192029143412e-07,
    'objective': 'reg:squarederror',
    'tree_method': 'hist',
    'early_stopping_rounds': 50
}

results = {}

for scenario in ['no_cme', 'with_cme']:
    print(f"\n--- Training Scenario: {scenario} ---")

    train_idx_list = data_split.loc[scenario, 'train']
    test_idx_list = data_split.loc[scenario, 'test']

    # Store metrics for each fold
    fold_rmse = []
    fold_mae = []
    fold_r2 = []
    fold_cc = []

    # Iterate through the cross-validation folds
    for fold_num in range(len(train_idx_list)):

        train_idx = train_idx_list[fold_num]
        test_idx = test_idx_list[fold_num]

        # Sanity check: Ensure absolutely no leakage between train and test in this fold
        assert len(train_idx.intersection(test_idx)) == 0, f"Leakage detected in fold {fold_num}!"

        X_train, Y_train = X.loc[train_idx].copy(), Y.loc[train_idx]
        X_test, Y_test = X.loc[test_idx].copy(), Y.loc[test_idx]

        # Clean column names for XGBoost
        X_train.columns = [str(col).replace('[', '').replace(']', '').replace('<', '').replace('>', '')
                           for col in X_train.columns]
        X_test.columns = X_train.columns

        model = xgb.XGBRegressor(**xgb_params)
        model.fit(
            X_train, Y_train,
            eval_set=[(X_test, Y_test)],
            verbose=False
        )

        preds = model.predict(X_test)

        rmse = np.sqrt(mean_squared_error(Y_test, preds))
        mae = mean_absolute_error(Y_test, preds)
        r2 = r2_score(Y_test, preds)

        # CC = Pearson correlation coefficient (same idea as in your evaluation.py)
        # guard against near-constant predictions
        if np.std(preds) < 1e-8:
            cc = np.nan
        else:
            cc = np.corrcoef(np.asarray(Y_test), np.asarray(preds))[0, 1]

        fold_rmse.append(rmse)
        fold_mae.append(mae)
        fold_r2.append(r2)
        fold_cc.append(cc)

        print(f"Fold {fold_num+1} - RMSE: {rmse:.2f} km/s | MAE: {mae:.2f} km/s | R2: {r2:.3f} | CC: {cc:.3f}")

    avg_rmse = float(np.nanmean(fold_rmse))
    avg_mae = float(np.nanmean(fold_mae))
    avg_r2 = float(np.nanmean(fold_r2))
    avg_cc = float(np.nanmean(fold_cc))

    print(f">>> {scenario} Average - RMSE: {avg_rmse:.2f} km/s | MAE: {avg_mae:.2f} km/s | "
          f"R2: {avg_r2:.3f} | CC: {avg_cc:.3f}")

    results[scenario] = {
        'rmse_avg': avg_rmse,
        'mae_avg': avg_mae,
        'r2_avg': avg_r2,
        'cc_avg': avg_cc
    }